# SUBFASE 4: Feature Engineering para SVC Multihorizonte

Objetivo: construir variables tecnicas y objetivos direccionales por horizonte para alimentar la fase de modelado clasico.

Flujo metodologico:
1. Partir del dataset limpio (Fase 1).
2. Construir serie por dia de trading (sin imputacion artificial).
3. Generar retornos, indicadores y lags con trazabilidad.
4. Construir targets y_h con horizonte futuro (h=1, h=5, h=20).
5. Exportar matrices consistentes por emisor/horizonte para Walk-Forward en Fase 3.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from bvg_core.config import COMPANIES
from bvg_core.data import aggregate_trade_day
from bvg_core.features import build_features_for_company

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 260)

In [2]:
INPUT_PATH = Path('../data/processed/BVG_Acciones_limpio.csv')
OUTPUT_MASTER = Path('../data/processed/BVG_features_svc_master.csv')
OUTPUT_DICT = Path('../data/processed/BVG_features_svc_dictionary.csv')

empresas_objetivo = COMPANIES
fecha_col, emisor_col, precio_col = 'FECHA NEGOCIACIÓN', 'EMISOR', 'PRECIO'
acciones_col = 'NÚMERO DE ACCIONES'
valor_efecto_col = 'VALOR EFECTO'
HORIZONS = [1, 5, 20]

In [3]:
df = pd.read_csv(INPUT_PATH)
df[fecha_col] = pd.to_datetime(df[fecha_col], errors='coerce')
df = df.dropna(subset=[fecha_col, emisor_col, precio_col]).copy()
df = df[df[emisor_col].isin(empresas_objetivo)].copy()

print('Shape filtrado:', df.shape)
display(df.head())

Shape filtrado: (18188, 11)


,FECHA NEGOCIACIÓN,TÍTULO,EMISOR,NÚMERO DE ACCIONES,V. NOM. UNITARIO,PRECIO,VALOR NOMINAL,VALOR EFECTO,CASA COMPRADORA,CASA VENDEDORA,BOLSA
0,2019-01-02,ACCIONES,BANCO GUAYAQUIL S.A.,2000.0,1.0,0.96,2000.0,1920.00,PLUSBURSÁTIL,SANTA FE,BVG
1,2019-01-03,ACCIONES,BANCO GUAYAQUIL S.A.,11069.0,1.0,0.96,11069.0,10626.24,SILVERCROSS,SILVERCROSS,BVG
2,2019-01-03,ACCIONES,BANCO GUAYAQUIL S.A.,20035.0,1.0,0.96,20035.0,19233.60,SILVERCROSS,SILVERCROSS,BVG
3,2019-01-14,ACCIONES,BANCO GUAYAQUIL S.A.,1978.0,1.0,0.96,1978.0,1898.88,SANTA FE,SANTA FE,BVG
4,2019-01-14,ACCIONES,BANCO GUAYAQUIL S.A.,13022.0,1.0,0.95,13022.0,12370.90,SANTA FE,ORION,BVG


In [4]:
# Agregacion diaria por emisor (serie por eventos reales de trading)
def build_daily_panel(df_input: pd.DataFrame) -> pd.DataFrame:
    daily_rows = []
    for empresa in empresas_objetivo:
        d_emp = df_input[df_input[emisor_col] == empresa].copy()
        d_day = d_emp.groupby(fecha_col).apply(aggregate_trade_day).reset_index()
        d_day = d_day.rename(columns={fecha_col: 'fecha'})
        d_day['empresa'] = empresa
        d_day = d_day.sort_values('fecha').reset_index(drop=True)
        daily_rows.append(d_day)
    return pd.concat(daily_rows, ignore_index=True)

df_daily = build_daily_panel(df)
print('Shape diario:', df_daily.shape)
display(df_daily.head())

Shape diario: (2871, 7)


,fecha,close_last,close_vwap,volume_shares_day,turnover_value_day,n_trades_day,empresa
0,2019-01-02,2.44,2.441314,7610.0,18578.40,7.0,CORPORACION FAVORITA C.A.
1,2019-01-03,2.45,2.445289,3812.0,9321.44,6.0,CORPORACION FAVORITA C.A.
2,2019-01-04,2.44,2.440000,1236.0,3015.84,1.0,CORPORACION FAVORITA C.A.
3,2019-01-07,2.45,2.441714,9343.0,22812.93,8.0,CORPORACION FAVORITA C.A.
4,2019-01-09,2.45,2.447877,14379.0,35198.03,9.0,CORPORACION FAVORITA C.A.


## Feature engineering (set comun para todos los horizontes)

Familias de features incluidas:
- Memoria corta y momentum: `ret_lag_*`, `mom_*`.
- Regimen de volatilidad: `vol_5`, `vol_10`, `regime_vol_ratio`.
- Tendencia/precio relativo: `ma_5`, `ma_10`, `ma_gap`, `price_vs_ma10`.
- Indicador tecnico robusto: `rsi_14`.
- Liquidez/microestructura: `turnover_log1p`, `volume_log1p`, `avg_trade_size`, `amihud_5`, `days_since_trade`.

Diseno anti-leakage:
- Se usa `shift(1)` en retornos/ventanas para que la fila en fecha t use solo informacion observada hasta t-1.

In [5]:
feat_rows = []
for empresa in empresas_objetivo:
    d_emp = df_daily[df_daily['empresa'] == empresa].copy()
    feat_rows.append(build_features_for_company(d_emp, horizons=HORIZONS))

df_feat = pd.concat(feat_rows, ignore_index=True)
df_feat = df_feat.sort_values(['empresa', 'fecha']).reset_index(drop=True)

print('Shape con features:', df_feat.shape)
display(df_feat.head())

Shape con features: (2871, 34)


,fecha,close_last,close_vwap,volume_shares_day,turnover_value_day,n_trades_day,empresa,ret_1d,ret_lag_1,ret_lag_2,ret_lag_3,mom_3,mom_5,mom_10,vol_5,vol_10,regime_vol_ratio,ma_5,ma_10,ma_gap,price_vs_ma10,rsi_14,turnover_log1p,volume_log1p,avg_trade_size,avg_trade_size_log1p,amihud_5,days_since_trade,ret_fwd_h1,target_up_h1,ret_fwd_h5,target_up_h5,ret_fwd_h20,target_up_h20
0,2019-01-02,0.96,0.960000,2000.0,1920.00,1.0,BANCO GUAYAQUIL S.A.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.560601,7.601402,2000.000000,7.601402,NaN,1.0,0.000000,0.0,0.000000,0.0,0.040822,1.0
1,2019-01-03,0.96,0.960000,31104.0,29859.84,2.0,BANCO GUAYAQUIL S.A.,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.304303,10.345124,15552.000000,9.652009,NaN,1.0,0.000000,0.0,0.010363,1.0,0.040822,1.0
2,2019-01-14,0.96,0.952217,16731.0,15931.54,3.0,BANCO GUAYAQUIL S.A.,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.676119,9.725078,5577.000000,8.626586,NaN,11.0,0.000000,0.0,0.010363,1.0,0.040822,1.0
3,2019-01-15,0.96,0.960000,2385.0,2289.60,3.0,BANCO GUAYAQUIL S.A.,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.736569,7.777374,795.000000,6.679599,NaN,1.0,0.010363,1.0,0.010363,1.0,0.040822,1.0
4,2019-01-16,0.97,0.958346,80864.0,77495.71,9.0,BANCO GUAYAQUIL S.A.,0.010363,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.257991,11.300536,8984.888889,9.103411,NaN,1.0,-0.010363,0.0,-0.010363,0.0,0.030459,1.0


In [6]:
# Control de calidad del set de features
feature_cols = [
    'ret_lag_1', 'ret_lag_2', 'ret_lag_3',
    'mom_3', 'mom_5', 'mom_10',
    'vol_5', 'vol_10', 'regime_vol_ratio',
    'ma_5', 'ma_10', 'ma_gap', 'price_vs_ma10',
    'rsi_14',
    'turnover_log1p', 'volume_log1p',
    'avg_trade_size_log1p',
    'amihud_5', 'days_since_trade'
]

target_cols = ['target_up_h1', 'target_up_h5', 'target_up_h20']

# Dataset apto para modelado: features completas
df_model_ready = df_feat.dropna(subset=feature_cols).copy()

rows_balance = []
for empresa in empresas_objetivo:
    d = df_model_ready[df_model_ready['empresa'] == empresa].copy()
    for tcol in target_cols:
        y = d[tcol].dropna()
        if len(y) == 0:
            rows_balance.append({'empresa': empresa, 'target': tcol, 'n': 0, 'p_up': np.nan, 'p_down': np.nan})
            continue
        p_up = float((y == 1).mean())
        rows_balance.append({
            'empresa': empresa,
            'target': tcol,
            'n': int(len(y)),
            'p_up': p_up,
            'p_down': 1.0 - p_up
        })

balance_df = pd.DataFrame(rows_balance)
display(balance_df)

,empresa,target,n,p_up,p_down
0,CORPORACION FAVORITA C.A.,target_up_h1,1687,0.317131,0.682869
1,CORPORACION FAVORITA C.A.,target_up_h5,1683,0.365419,0.634581
2,CORPORACION FAVORITA C.A.,target_up_h20,1668,0.413070,0.586930
3,BANCO GUAYAQUIL S.A.,target_up_h1,1152,0.325521,0.674479
4,BANCO GUAYAQUIL S.A.,target_up_h5,1148,0.441638,0.558362
5,BANCO GUAYAQUIL S.A.,target_up_h20,1133,0.528685,0.471315


In [7]:
# Exportacion CSV maestro para Fase 5
export_cols = [
    'fecha', 'empresa',
    'close_last', 'close_vwap',
    'volume_shares_day', 'turnover_value_day', 'n_trades_day',
] + feature_cols + [
    'ret_fwd_h1', 'ret_fwd_h5', 'ret_fwd_h20',
    'target_up_h1', 'target_up_h5', 'target_up_h20'
]

df_export = df_model_ready[export_cols].copy().sort_values(['empresa', 'fecha']).reset_index(drop=True)

df_export.to_csv(OUTPUT_MASTER, index=False)

# Diccionario rapido para trazabilidad de fase
dict_rows = [
    {'feature': 'ret_lag_1..3', 'familia': 'memoria_corta', 'justificacion': 'dependencia temporal local'},
    {'feature': 'mom_3, mom_5, mom_10', 'familia': 'momentum', 'justificacion': 'ventanas deslizantes en literatura SVM/SVR'},
    {'feature': 'vol_5, vol_10, regime_vol_ratio', 'familia': 'volatilidad', 'justificacion': 'FTS heterocedastica'},
    {'feature': 'ma_5, ma_10, ma_gap, price_vs_ma10', 'familia': 'tendencia', 'justificacion': 'senal de estructura local de precio'},
    {'feature': 'rsi_14', 'familia': 'tecnico', 'justificacion': 'oscilador usado frecuentemente en estudios de mercado'},
    {'feature': 'turnover_log1p, volume_log1p, avg_trade_size_log1p, amihud_5, days_since_trade', 'familia': 'liquidez_microestructura', 'justificacion': 'mercado BVG con episodios de iliquidez'}
]
dict_df = pd.DataFrame(dict_rows)
dict_df.to_csv(OUTPUT_DICT, index=False)

print(f'CSV maestro exportado en: {OUTPUT_MASTER.as_posix()}')
print('Shape CSV maestro:', df_export.shape)

print(f'Diccionario exportado en: {OUTPUT_DICT.as_posix()}')
display(df_export.head())

CSV maestro exportado en: ../data/processed/BVG_features_svc_master.csv
Shape CSV maestro: (2841, 32)
Diccionario exportado en: ../data/processed/BVG_features_svc_dictionary.csv


,fecha,empresa,close_last,close_vwap,volume_shares_day,turnover_value_day,n_trades_day,ret_lag_1,ret_lag_2,ret_lag_3,mom_3,mom_5,mom_10,vol_5,vol_10,regime_vol_ratio,ma_5,ma_10,ma_gap,price_vs_ma10,rsi_14,turnover_log1p,volume_log1p,avg_trade_size_log1p,amihud_5,days_since_trade,ret_fwd_h1,ret_fwd_h5,ret_fwd_h20,target_up_h1,target_up_h5,target_up_h20
0,2019-02-13,BANCO GUAYAQUIL S.A.,0.97,0.97,35000.0,33950.0,1.0,0.040822,-0.040822,0.000000,3.469447e-17,4.082199e-02,0.030459,0.034154,0.024103,1.416994,0.984,0.975,0.009,0.025641,62.452022,10.432674,10.463132,10.463132,0.000016,1.0,0.030459,0.030459,-0.010363,1.0,1.0,0.0
1,2019-02-19,BANCO GUAYAQUIL S.A.,1.00,1.00,2046.0,2046.0,6.0,-0.030459,0.040822,-0.040822,-3.045921e-02,1.036279e-02,0.010363,0.038424,0.026100,1.472155,0.986,0.976,0.010,-0.006148,52.665650,7.624131,7.624131,5.834811,0.000016,6.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0
2,2019-02-20,BANCO GUAYAQUIL S.A.,1.00,1.00,1500.0,1500.0,1.0,0.030459,-0.030459,0.040822,4.082199e-02,1.144917e-16,0.030459,0.036015,0.027627,1.303608,0.986,0.979,0.007,0.021450,59.078190,7.313887,7.313887,7.313887,0.000019,1.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0
3,2019-02-21,BANCO GUAYAQUIL S.A.,1.00,1.00,591494.0,591494.0,14.0,0.000000,0.030459,-0.030459,7.979728e-17,1.144917e-16,0.030459,0.036015,0.027627,1.303608,0.986,0.982,0.004,0.018330,59.078190,13.290409,13.290409,10.651373,0.000019,1.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0
4,2019-02-26,BANCO GUAYAQUIL S.A.,1.00,1.00,2771.0,2771.0,1.0,0.000000,0.000000,0.030459,3.045921e-02,4.082199e-02,0.030459,0.028234,0.027627,1.021964,0.994,0.985,0.009,0.015228,57.100950,7.927324,7.927324,7.927324,0.000003,5.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0


## Conclusiones de subfase

### Objetivos realizados

- Se construyó y dejó aislado el flujo de feature engineering para clasificación direccional con SVC, dejando un dataset final listo para modelado.

- Se generó el CSV maestro de features y su diccionario de trazabilidad. Las variables diseñadas se organizaron en familias con sentido financiero y utilidad para clasificación direccional:

- Volatilidad y régimen: vol_5, vol_10, regime_vol_ratio.
- Memoria corta y momentum: ret_lag_1, ret_lag_2, ret_lag_3, mom_3, mom_5, mom_10.
- Tendencia de precio: ma_5, ma_10, ma_gap, price_vs_ma10.
- Indicador técnico: rsi_14.
- Liquidez y microestructura: turnover_log1p, volume_log1p, avg_trade_size_log1p, amihud_5, days_since_trade.

Además, se mantuvo la coherencia multihorizonte (h=1, h=5, h=20): se utilizó un mismo set de features para todos los horizontes y se definieron targets separados por horizonte.